# Differentiated duopoly simulator

Run all cells, then use the sliders to explore the model. The parameters are
illustrative and the prices are expressed in model units.

The table and best-response intersection show the equilibrium at the selected
parameters. The comparison plots vary only relative-price sensitivity, s.
The dotted line marks the selected value of s.

Select **Symmetric demand** to set δ = 0. Use **Reset to F1** to restore the
illustrative scenario used in Figure F1 of the project.

In [ ]:
import pandas as pd

from duopoly.model import Params, equilibrium


def equilibrium_table(params: Params) -> pd.DataFrame:
    eq = equilibrium(params)

    return pd.DataFrame(
        {
            "Demand intercept": [params.a + params.delta, params.a - params.delta],
            "Price": eq.prices,
            "Quantity": eq.quantities,
            "Margin": eq.margins,
            "Profit": eq.profits,
            "Share (%)": 100 * eq.shares,
        },
        index=pd.Index(["Uber", "Lyft"], name="Firm"),
    )


baseline = Params(a=7, delta=-2, k=1, s=1, c=4)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
from duopoly.plots import plot_best_responses, plot_comparative_statics


def slider(label, value, minimum, maximum):
    return widgets.FloatSlider(
        description=label,
        value=value,
        min=minimum,
        max=maximum,
        step=0.1,
        continuous_update=False,
        style={"description_width": "initial"},
    )


controls = {
    "a": slider("Base demand (a)", baseline.a, 1, 20),
    "delta": slider("Demand asymmetry (δ)", baseline.delta, -5, 5),
    "k": slider("Base price sensitivity (k)", baseline.k, 0.1, 3),
    "s": slider("Relative-price sensitivity (s)", baseline.s, 0, 6),
    "c": slider("Marginal cost (c)", baseline.c, 0, 8),
}


s_grid = np.linspace(controls["s"].min, controls["s"].max, 121)


def show_equilibrium(a, delta, k, s, c):
    params = Params(a=a, delta=delta, k=k, s=s, c=c)
    try:
        table = equilibrium_table(params)
    except ValueError as error:
        print(f"Invalid scenario: {error}")
        return

    display(table.round(3))

    responses = plot_best_responses(params)
    display(responses)
    plt.close(responses)

    comparison = plot_comparative_statics(params, s_grid, show_shares=(delta != 0))
    for ax in comparison.axes:
        ax.axvline(s, color="gray", linestyle=":", linewidth=1)
    display(comparison)
    plt.close(comparison)


symmetry = widgets.Checkbox(value=False, description="Symmetric demand (δ = 0)")
reset_button = widgets.Button(description="Reset to F1")


def toggle_symmetry(change):
    delta_slider = controls["delta"]
    delta_slider.disabled = change["new"]
    if change["new"]:
        delta_slider.value = 0.0


def reset_to_baseline(_):
    symmetry.value = False
    for name, control in controls.items():
        control.value = getattr(baseline, name)


symmetry.observe(toggle_symmetry, names="value")
reset_button.on_click(reset_to_baseline)

output = widgets.interactive_output(show_equilibrium, controls)
actions = widgets.HBox([symmetry, reset_button])
display(widgets.VBox([actions, *controls.values(), output]))